# Near-Earth Asteroid Approaches

*Reproducible research notebook from [Flarient](https://flarient.com) — the space weather intelligence platform.*

**About this notebook:** This notebook is part of the [Flarient Research Notebooks](https://github.com/flarientglobal/flarient-notebooks) collection. It uses public data from NOAA SWPC, NASA, and the Flarient API.


## 1. Introduction

Near-Earth Objects (NEOs) are asteroids and comets that pass close to Earth. In this notebook, we'll explore NEO data from NASA's NeoWS API.


In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (12, 6)


## 2. Fetching NEO Data

We'll use NASA's NeoWS (Near Earth Object Web Service) API.


In [ ]:
# Fetch NEOs for the next 7 days
# Get a free API key at https://api.nasa.gov/
NASA_API_KEY = "DEMO_KEY"  # Replace with your own key for higher rate limits

url = f"https://api.nasa.gov/neo/rest/v1/feed?start_date=2026-08-19&end_date=2026-08-26&api_key={NASA_API_KEY}"
response = requests.get(url, timeout=30)
data = response.json()

# Parse NEO data
neos = []
for date, objects in data.get('near_earth_objects', {}).items():
    for obj in objects:
        neos.append({
            'name': obj['name'],
            'date': date,
            'diameter_min_m': float(obj['estimated_diameter']['meters']['estimated_diameter_min']),
            'diameter_max_m': float(obj['estimated_diameter']['meters']['estimated_diameter_max']),
            'miss_distance_km': float(obj['close_approach_data'][0]['miss_distance']['kilometers']),
            'velocity_kmh': float(obj['close_approach_data'][0]['relative_velocity']['kilometers_per_hour']),
            'hazardous': obj['is_potentially_hazardous_asteroid'],
        })

df = pd.DataFrame(neos)
print(f"Found {len(df)} NEOs in the next 7 days")
df.head(10)


## 3. Analysing NEO Sizes

Let's look at the distribution of NEO sizes.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df['diameter_max_m'], bins=30, color='#6366f1', alpha=0.7, edgecolor='#22d3ee')
ax.set_xlabel('Estimated Diameter (m)')
ax.set_ylabel('Count')
ax.set_title('Near-Earth Object Size Distribution — Source: NASA NeoWS')
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig('neo_sizes.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Largest NEO: {df.loc[df['diameter_max_m'].idxmax(), 'name']} ({df['diameter_max_m'].max():.0f}m)")
print(f"Smallest NEO: {df.loc[df['diameter_min_m'].idxmin(), 'name']} ({df['diameter_min_m'].min():.1f}m)")


## 4. Miss Distance Analysis

How close do these objects come to Earth?


In [ ]:
# Convert miss distance to AU for easier interpretation
df['miss_distance_au'] = df['miss_distance_km'] / 149597870.7

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#ef4444' if h else '#22d3ee' for h in df['hazardous']]
ax.barh(range(len(df)), df['miss_distance_au'], color=colors, alpha=0.7)
ax.set_yticks(range(len(df)))
ax.set_yticklabels([n.replace('(', '').replace(')', '')[:20] for n in df['name']])
ax.set_xlabel('Miss Distance (AU)')
ax.set_title('NEO Miss Distances — Red = Potentially Hazardous')
ax.grid(True, alpha=0.2, axis='x')
plt.tight_layout()
plt.savefig('neo_distances.png', dpi=150, bbox_inches='tight')
plt.show()


## 5. Potentially Hazardous Asteroids

Let's identify which NEOs are classified as potentially hazardous.


In [ ]:
hazardous = df[df['hazardous']]
print(f"Potentially Hazardous Asteroids: {len(hazardous)}")
if len(hazardous) > 0:
    print("\nDetails:")
    for _, h in hazardous.iterrows():
        print(f"  {h['name']}: {h['diameter_max_m']:.0f}m diameter, {h['miss_distance_au']:.4f} AU miss distance")


## 6. Velocity Analysis

How fast are these objects moving?


In [ ]:
# Convert to km/s
df['velocity_kms'] = df['velocity_kmh'] / 3600

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(df['velocity_kms'], df['miss_distance_au'], 
           c=df['diameter_max_m'], cmap='viridis', s=50, alpha=0.7)
plt.colorbar(ax.collections[0], label='Diameter (m)')
ax.set_xlabel('Velocity (km/s)')
ax.set_ylabel('Miss Distance (AU)')
ax.set_title('NEO Velocity vs Miss Distance — Source: NASA NeoWS')
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig('neo_velocity.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. Tracking NEOs with Flarient

Flarient provides real-time NEO tracking and alerts. Visit:
- [flarient.com/near-earth-objects](https://flarient.com/near-earth-objects) for live NEO data
- [flarient.com/space-events](https://flarient.com/space-events) for asteroid approach events
- [flarient.com/widgets](https://flarient.com/widgets) to embed NEO widgets on your site


## 8. Conclusion

This notebook demonstrated:
1. Fetching NEO data from NASA's NeoWS API
2. Analysing size, distance, and velocity distributions
3. Identifying potentially hazardous asteroids
4. Visualising NEO close approaches

For real-time NEO monitoring and alerts, visit [flarient.com](https://flarient.com).


---

## About Flarient

[Flarient](https://flarient.com) is a space weather intelligence platform providing real-time data, forecasts, and community-driven observations. Visit [flarient.com](https://flarient.com) for live space weather conditions, aurora forecasts, and more.

## License

MIT — This notebook is open source. [View on GitHub](https://github.com/flarientglobal/flarient-notebooks).
